# ETF event study and backtest (CMC)

Loads cached OHLCV, computes CAR around ETF milestones, and runs the simple pre-event entry backtest.

In [ ]:
from pathlib import Path
import pandas as pd
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from cmc_data import get_ohlcv_daily, MissingAPIKeyError
from event_study import compute_event_windows, summarize_car, backtest_event_strategy, trade_stats

events = pd.read_csv(ROOT / 'data' / 'etf_events.csv')
events['event_date'] = pd.to_datetime(events['event_date']).dt.date
assets = sorted(set(events['asset']) | {'BTC'})
prices = {}
for asset in assets:
    sym = asset if asset != 'MIX' else 'BTC'
    try:
        prices[sym] = get_ohlcv_daily(sym)
    except MissingAPIKeyError as exc:
        cache_path = ROOT / 'data' / f'prices_{sym.lower()}_daily.csv'
        if cache_path.exists():
            df = pd.read_csv(cache_path, parse_dates=['date'])
            df['date'] = df['date'].dt.date
            prices[sym] = df
        else:
            raise exc
print('Loaded assets:', list(prices))

In [ ]:
results = compute_event_windows(prices, events, window=(-10, 10), market_symbol='BTC')
summary = summarize_car(results.values())
summary.head()

In [ ]:
trades = backtest_event_strategy(prices, events, hold_days=1, entry_offset=-1, volatility_widening=True)
trade_stats(trades)